# ITAI 1371 Midterm: Netflix Movies Data Preparation
**Project:** Predict whether a Netflix *movie* is kid-safe (`kid_safe`) from its metadata.
**Dataset:** Kaggle `shivamb/netflix-shows` (movies only).
**Rule followed:** the raw file is never edited by hand. Every change below is done in Python code.

**Pipeline order (and why)**
1. Load the raw data and keep movies only
2. Inspect raw data: missing values, odd values
3. Build the `kid_safe` target from `rating`
4. Engineer features (row-by-row, so there is no leakage)
5. **Split 70/30 before anything is learned from the data**
6. Fill missing values (statistics learned from the training set only)
7. Normalize skewed numbers, then scale ranges
8. Encode categories (one-hot, multi-hot, ordinal)
9. Balance classes (training set only)
10. Save the outputs and discuss the ML problem

In [ ]:
# Step 0: setup
import warnings, os, glob
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import PowerTransformer, MinMaxScaler, OneHotEncoder, OrdinalEncoder, MultiLabelBinarizer
from sklearn.utils import resample

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 50, "display.width", 200)
RANDOM_STATE = 42
plt.rcParams["figure.dpi"] = 100

## Step 1. Load the raw data and keep movies only

In [ ]:
# Download the public Kaggle dataset in code (no manual download or editing).
# If kagglehub is unavailable, upload netflix_titles.csv to the Colab session and it will be used instead.
try:
    import kagglehub
    path = kagglehub.dataset_download("shivamb/netflix-shows")
    csv_path = glob.glob(os.path.join(path, "*.csv"))[0]
except Exception as e:
    print("kagglehub failed, falling back to a local file:", e)
    csv_path = "netflix_titles.csv"

raw = pd.read_csv(csv_path)
print("Source file:", csv_path)
print("Raw shape (movies + TV shows):", raw.shape)
print(raw["type"].value_counts())
raw.head()

In [ ]:
# Keep movies only (TV shows removed in code)
movies = raw[raw["type"].str.strip() == "Movie"].copy()
print(f"Rows before filter: {len(raw):,} | Movies kept: {len(movies):,} | TV shows removed: {len(raw) - len(movies):,}")
assert len(movies) >= 2000, "Dataset must have at least 2,000 rows"
print("Meets the 2,000-row minimum:", len(movies) >= 2000)
movies.head()

## Step 2. Inspect the raw movie data (BEFORE cleanup)

In [ ]:
print(movies.dtypes, "\n")
missing_raw = movies.isna().sum()
missing_pct = (missing_raw / len(movies) * 100).round(2)
missing_table = pd.DataFrame({"missing": missing_raw, "percent": missing_pct}).sort_values("missing", ascending=False)
print(missing_table)

fig, ax = plt.subplots(figsize=(8, 4))
shown = missing_table[missing_table["missing"] > 0]
ax.barh(shown.index[::-1], shown["percent"][::-1], color="tab:red")
ax.set_xlabel("% of movie rows missing")
ax.set_title("BEFORE: missing values by column (movies only)")
for i, v in enumerate(shown["percent"][::-1]):
    ax.text(v + 0.3, i, f"{v:.1f}%", va="center")
plt.tight_layout(); plt.show()

In [ ]:
# Look for values that are present but wrong. In this dataset a few movies have a run time
# typed into the 'rating' column and an empty 'duration'.
movies["rating"] = movies["rating"].astype("string").str.strip()
misplaced = movies["rating"].str.contains("min", na=False)
print("Rows with a run time stored in 'rating':", misplaced.sum())
print(movies.loc[misplaced, ["title", "rating", "duration"]])

# Fix in code: move the value to duration, and treat the rating as missing
movies.loc[misplaced, "duration"] = movies.loc[misplaced, "rating"]
movies.loc[misplaced, "rating"] = pd.NA

print("\nRating counts after the fix:")
print(movies["rating"].value_counts(dropna=False))

## Step 3. Build the `kid_safe` target from `rating`

**Decision (documented):**
- `kid_safe = 1` for G, PG, TV-G, TV-PG, TV-Y, TV-Y7, TV-Y7-FV
- `kid_safe = 0` for PG-13, R, NC-17, TV-14, TV-MA
- `NR`, `UR`, and missing ratings have **no known maturity label**. They are described as "unrated" and are **not** treated as statistical outliers. They are excluded from the modeling table because we cannot assign them a true label.

The `rating` column is used only to build the target and is **not** a model input.

In [ ]:
KID_SAFE = {"G", "PG", "TV-G", "TV-PG", "TV-Y", "TV-Y7", "TV-Y7-FV"}
MATURE = {"PG-13", "R", "NC-17", "TV-14", "TV-MA"}

def make_target(r):
    if pd.isna(r): return np.nan
    if r in KID_SAFE: return 1
    if r in MATURE: return 0
    return np.nan   # NR, UR, or anything unrecognised

movies["kid_safe"] = movies["rating"].map(make_target)

unrated = movies["kid_safe"].isna()
print("Unrated / unknown-rating movies:", unrated.sum())
print(movies.loc[unrated, "rating"].value_counts(dropna=False), "\n")

fig, ax = plt.subplots(figsize=(9, 4))
order = movies["rating"].value_counts(dropna=False)
labels_r = [str(r) if pd.notna(r) else "missing" for r in order.index]
colors = ["tab:green" if r in KID_SAFE else "tab:orange" if r in MATURE else "tab:gray" for r in labels_r]
ax.bar(labels_r, order.values, color=colors)
ax.set_title("BEFORE: raw rating counts (green = kid_safe, orange = mature, gray = unrated)")
ax.set_ylabel("movies"); plt.xticks(rotation=45); plt.tight_layout(); plt.show()

model_df = movies.loc[~unrated].copy()
model_df["kid_safe"] = model_df["kid_safe"].astype(int)
print(f"Movies kept for modeling: {len(model_df):,} (dropped {unrated.sum()} unrated)")
print(model_df["kid_safe"].value_counts().rename({1: "kid_safe", 0: "mature"}))

## Step 4. Feature engineering
These features are computed row by row from a single movie's own information, so they cannot leak information between train and test.
Missing text fields get a clear `*_missing` flag so the missingness itself is documented and available to a model.

In [ ]:
def count_items(s):
    return s.fillna("").apply(lambda x: len([i for i in x.split(",") if i.strip()]))

df = model_df.copy()
df["duration_min"] = df["duration"].str.extract(r"(\d+)")[0].astype(float)

added = pd.to_datetime(df["date_added"].astype("string").str.strip(), format="mixed", errors="coerce")
df["year_added"] = added.dt.year
df["month_added"] = added.dt.month
df["years_to_netflix"] = (df["year_added"] - df["release_year"]).clip(lower=0)

df["n_cast"] = count_items(df["cast"])
df["n_directors"] = count_items(df["director"])
df["n_countries"] = count_items(df["country"])
df["n_genres"] = count_items(df["listed_in"])
df["director_missing"] = df["director"].isna().astype(int)
df["cast_missing"] = df["cast"].isna().astype(int)
df["country_missing"] = df["country"].isna().astype(int)

df["desc_words"] = df["description"].fillna("").str.split().str.len()
df["title_len"] = df["title"].fillna("").str.len()

df["primary_country"] = df["country"].str.split(",").str[0].str.strip()
df["genres"] = df["listed_in"].fillna("").apply(lambda x: [g.strip() for g in x.split(",") if g.strip()])

num_cols = ["release_year", "duration_min", "year_added", "month_added", "years_to_netflix",
            "n_cast", "n_directors", "n_countries", "n_genres", "desc_words", "title_len"]
flag_cols = ["director_missing", "cast_missing", "country_missing"]

print("Engineered columns:", ["duration_min", "year_added", "month_added", "years_to_netflix",
      "n_cast", "n_directors", "n_countries", "n_genres", "desc_words", "title_len",
      "primary_country", "genres", "*_missing flags"])
df[["title", "duration", "duration_min", "date_added", "year_added", "years_to_netflix", "n_cast", "n_genres", "primary_country", "kid_safe"]].head(8)

## Step 5. Train/test split (70/30, stratified)
We split **before** filling, scaling, encoding, or balancing so that nothing is learned from the test set.
The test set stays untouched and keeps its natural class mix, because it represents unseen data.

In [ ]:
feature_cols = num_cols + flag_cols + ["primary_country", "genres"]   # note: 'rating' is NOT here
X = df[feature_cols]
y = df["kid_safe"]
assert "rating" not in X.columns and "kid_safe" not in X.columns

Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.30, stratify=y, random_state=RANDOM_STATE)
print(f"Train: {len(Xtr):,} rows | Test: {len(Xte):,} rows")
print("Train kid_safe share:", round(ytr.mean(), 3), "| Test kid_safe share:", round(yte.mean(), 3))

## Step 6. Fill missing values (statistics learned from the training set only)

In [ ]:
before_missing = Xtr.isna().sum()
print("Missing values in TRAIN before filling:")
print(before_missing[before_missing > 0], "\n")
print("Sample rows with a missing value (BEFORE):")
display(Xtr[Xtr[num_cols + ["primary_country"]].isna().any(axis=1)].head(5)[num_cols[:5] + ["primary_country"]])

medians = Xtr[num_cols].median()          # learned from train only
Xtr_i, Xte_i = Xtr.copy(), Xte.copy()
for d in (Xtr_i, Xte_i):
    d[num_cols] = d[num_cols].fillna(medians)
    d["primary_country"] = d["primary_country"].fillna("Unknown")

after_missing = Xtr_i.isna().sum()
print("\nMissing values in TRAIN after filling (should all be 0):")
print(after_missing[after_missing > 0] if after_missing.sum() else "none")

cols = before_missing[before_missing > 0].index.tolist()
if cols:
    fig, ax = plt.subplots(figsize=(8, 4))
    w = 0.4; idx = np.arange(len(cols))
    ax.bar(idx - w/2, before_missing[cols], w, label="before", color="tab:red")
    ax.bar(idx + w/2, after_missing[cols], w, label="after", color="tab:green")
    ax.set_xticks(idx); ax.set_xticklabels(cols, rotation=30); ax.set_ylabel("missing cells (train)")
    ax.set_title("Missing values before vs after filling"); ax.legend(); plt.tight_layout(); plt.show()

print("\nSample rows (AFTER):")
display(Xtr_i[num_cols[:5] + ["primary_country"]].head(5))

**Decisions (documented):** numeric gaps are filled with the *training median* (robust to skew). Missing `primary_country` becomes its own category, `"Unknown"`, because missing country is information, not a random gap.
Director and cast gaps are already captured by the `*_missing` flags and `n_*` counts (0 people listed).

## Step 7. Outliers, normalization (shape), and scaling (range)
Unusually long or short movies and old release years are **real data, not errors**, so they are kept. We check them with the IQR rule, and then reduce skew and put every number on a 0 to 1 range.

In [ ]:
q1, q3 = Xtr_i[num_cols].quantile(0.25), Xtr_i[num_cols].quantile(0.75)
iqr = q3 - q1
outlier_counts = ((Xtr_i[num_cols] < q1 - 1.5 * iqr) | (Xtr_i[num_cols] > q3 + 1.5 * iqr)).sum()
skew = Xtr_i[num_cols].skew().round(2)
summary = pd.DataFrame({"skew": skew, "iqr_outliers": outlier_counts, "min": Xtr_i[num_cols].min(), "max": Xtr_i[num_cols].max()})
print(summary.sort_values("skew", key=abs, ascending=False))

fig, ax = plt.subplots(figsize=(6, 3.5))
ax.boxplot(Xtr_i["duration_min"], vert=False)
ax.set_title("duration_min: long tail is real (kept, not removed)"); ax.set_xlabel("minutes")
plt.tight_layout(); plt.show()

In [ ]:
# Normalization = change the SHAPE (reduce skew) with a Yeo-Johnson power transform, on strongly skewed columns only.
skewed = [c for c in num_cols if abs(Xtr_i[c].skew()) > 0.75]
print("Strongly skewed columns normalized:", skewed)

Xtr_n, Xte_n = Xtr_i.copy(), Xte_i.copy()
if skewed:
    pt = PowerTransformer(method="yeo-johnson", standardize=False).fit(Xtr_i[skewed])
    Xtr_n[skewed] = pt.transform(Xtr_i[skewed])
    Xte_n[skewed] = pt.transform(Xte_i[skewed])

# Scaling = change the RANGE to 0-1 with MinMaxScaler (fit on train only)
scaler = MinMaxScaler().fit(Xtr_n[num_cols])
Xtr_s, Xte_s = Xtr_n.copy(), Xte_n.copy()
Xtr_s[num_cols] = scaler.transform(Xtr_n[num_cols])
Xte_s[num_cols] = scaler.transform(Xte_n[num_cols])

print("\nSample numeric rows BEFORE normalization/scaling:")
display(Xtr_i[num_cols].head(5).round(2))
print("Sample numeric rows AFTER normalization/scaling:")
display(Xtr_s[num_cols].head(5).round(3))

# Before/after histograms for the most skewed columns
top = skew.abs().sort_values(ascending=False).index[:4].tolist()
fig, axes = plt.subplots(2, len(top), figsize=(4 * len(top), 6))
for j, c in enumerate(top):
    axes[0, j].hist(Xtr_i[c], bins=30, color="tab:blue"); axes[0, j].set_title(f"BEFORE: {c}\nskew={Xtr_i[c].skew():.2f}")
    axes[1, j].hist(Xtr_s[c], bins=30, color="tab:green"); axes[1, j].set_title(f"AFTER: {c}\nskew={Xtr_s[c].skew():.2f}")
plt.tight_layout(); plt.show()

**Note:** the test set is transformed with the transformers fitted on training data. A few test values can land slightly outside 0 to 1. That is expected and correct for unseen data.

## Step 8. Encoding categorical data

In [ ]:
# (a) One-hot encoding: primary_country, keeping the 10 most common countries in TRAIN and grouping the rest as 'Other'
top_countries = Xtr_i["primary_country"].value_counts().head(10).index
for d in (Xtr_i, Xte_i):
    d["country_grouped"] = d["primary_country"].where(d["primary_country"].isin(top_countries), "Other")

ohe = OneHotEncoder(handle_unknown="ignore", sparse_output=False).fit(Xtr_i[["country_grouped"]])
def ohe_frame(d):
    return pd.DataFrame(ohe.transform(d[["country_grouped"]]), index=d.index,
                        columns=[f"country_{c}" for c in ohe.categories_[0]])

# (b) Multi-hot encoding: a movie can have several genres
mlb = MultiLabelBinarizer().fit(Xtr["genres"])
def genre_frame(d):
    return pd.DataFrame(mlb.transform(d["genres"]), index=d.index, columns=[f"genre_{g}" for g in mlb.classes_])

# (c) Ordinal encoding: duration bucket has a natural order Short < Standard < Long
def bucket(d):
    return pd.cut(d["duration_min"], bins=[-np.inf, 90, 120, np.inf], labels=["Short", "Standard", "Long"]).astype(str)
ordenc = OrdinalEncoder(categories=[["Short", "Standard", "Long"]])
ordenc.fit(bucket(Xtr_i).to_frame("duration_bucket"))   # fit on train only
def ord_frame(d):
    b = bucket(d).to_frame("duration_bucket")
    return pd.DataFrame(ordenc.transform(b), index=d.index, columns=["duration_bucket_ord"])

print("Sample categorical rows BEFORE encoding:")
display(pd.concat([Xtr_i[["country_grouped"]], bucket(Xtr_i).rename("duration_bucket"),
                   Xtr["genres"].apply(lambda g: ", ".join(g)).rename("genres")], axis=1).head(5))

def assemble(d_scaled, d_imputed):
    return pd.concat([d_scaled[num_cols + flag_cols], ohe_frame(d_imputed), genre_frame(d_imputed), ord_frame(d_imputed)], axis=1)

Xtr_enc = assemble(Xtr_s, Xtr_i)       # transformers already fitted on train
Xte_enc = assemble(Xte_s, Xte_i)

print(f"\nFinal feature matrix: train {Xtr_enc.shape}, test {Xte_enc.shape}")
assert list(Xtr_enc.columns) == list(Xte_enc.columns)
assert Xtr_enc.isna().sum().sum() == 0 and Xte_enc.isna().sum().sum() == 0
print("Sample rows AFTER encoding:")
display(Xtr_enc.iloc[:5, :14].round(3))
print("Encoded column groups: numeric", len(num_cols), "| flags", len(flag_cols),
      "| country one-hot", len(ohe.categories_[0]), "| genre multi-hot", len(mlb.classes_), "| ordinal 1")

## Step 9. Class balancing (training set only)
We first **measure** the real class distribution, then balance only if it is clearly imbalanced.
Method: random oversampling of the minority class. This is transparent, works with one-hot data, and never creates fake fractional categories (unlike SMOTE on encoded categories).

In [ ]:
train = Xtr_enc.copy(); train["kid_safe"] = ytr
counts = train["kid_safe"].value_counts().sort_index()
minority_label = counts.idxmin(); majority_label = counts.idxmax()
ratio = counts.min() / counts.max()
print("Train class counts:\n", counts.rename({0: "mature (0)", 1: "kid_safe (1)"}))
print(f"Minority/majority ratio: {ratio:.2f}")

NEEDS_BALANCING = ratio < 0.8
print("Balancing needed:", NEEDS_BALANCING)

if NEEDS_BALANCING:
    minority = train[train["kid_safe"] == minority_label]
    majority = train[train["kid_safe"] == majority_label]
    minority_up = resample(minority, replace=True, n_samples=len(majority), random_state=RANDOM_STATE)
    train_bal = pd.concat([majority, minority_up]).sample(frac=1, random_state=RANDOM_STATE)
else:
    train_bal = train.copy()

fig, axes = plt.subplots(1, 3, figsize=(13, 4), sharey=True)
labels = ["mature (0)", "kid_safe (1)"]
axes[0].bar(labels, train["kid_safe"].value_counts().sort_index().values, color=["tab:orange", "tab:green"])
axes[0].set_title("TRAIN before balancing")
axes[1].bar(labels, train_bal["kid_safe"].value_counts().sort_index().values, color=["tab:orange", "tab:green"])
axes[1].set_title("TRAIN after balancing")
axes[2].bar(labels, yte.value_counts().sort_index().values, color=["tab:orange", "tab:green"])
axes[2].set_title("TEST (untouched)")
for a in axes:
    for p in a.patches: a.text(p.get_x() + p.get_width() / 2, p.get_height(), int(p.get_height()), ha="center", va="bottom")
plt.tight_layout(); plt.show()

print("Sample rows after balancing:")
display(train_bal.iloc[:5, :8].join(train_bal["kid_safe"]).round(3))

## Step 10. Final checks and saved outputs

In [ ]:
test = Xte_enc.copy(); test["kid_safe"] = yte

assert "rating" not in train_bal.columns
print("Rows: raw", len(raw), "| movies", len(movies), "| modeling", len(df),
      "| train (balanced)", len(train_bal), "| test", len(test))
print("Test kid_safe share (unchanged):", round(test["kid_safe"].mean(), 3))

train.to_csv("netflix_movies_train_unbalanced.csv", index=False)
train_bal.to_csv("netflix_movies_train_balanced.csv", index=False)
test.to_csv("netflix_movies_test.csv", index=False)
print("Saved: netflix_movies_train_unbalanced.csv, netflix_movies_train_balanced.csv, netflix_movies_test.csv")

# Optional in Colab: download the files
try:
    from google.colab import files
    for f in ["netflix_movies_train_balanced.csv", "netflix_movies_test.csv"]:
        files.download(f)
except ImportError:
    pass

## Step 11. How this dataset solves an ML problem

**Problem:** binary classification. Given a Netflix movie's metadata (genres, country, run time, release year, when it was added, cast and director counts, description length), predict whether it is **kid-safe** (`kid_safe = 1`) or mature (`0`).

**Why it is useful:** a streaming service could use the model to flag likely kid-friendly titles for a kids profile, or to suggest a maturity category for titles that arrive with no rating.

**Why the target is not a leak:** the `rating` column is used only to *create* the label and is removed from the inputs, so models must learn from content features instead.

**Planned models for the finals:** logistic regression (baseline), k-nearest neighbors, decision tree, random forest, gradient boosting.

**Evaluation plan:** because kid-safe movies are the minority, accuracy alone can look good while missing them. We will report precision, recall, F1, and ROC-AUC on the untouched 30% test set, and review the confusion matrix. For this use case, labeling a mature movie as kid-safe (false positive) is the costlier mistake, so precision on `kid_safe` matters.

**Known limits:** genre tags such as "Children & Family Movies" are very informative, so models may lean on them heavily. Unrated movies were excluded, so the model has not seen them. The text `description` is not yet used; TF-IDF features are an option for the finals.

## Reflection and contribution journal (required)
| Group member | Work completed | Date |
|---|---|---|
| | | |
| | | |
| | | |